# 01 Data Cleaning

This notebook cleans the LuminaTech sales data in a simple step-by-step way. The code is written close to the original assignment style so each transformation is easy to explain.


## Import Libraries

We only need pandas for data loading, inspection, cleaning, and saving.


In [1]:
import pandas as pd
import os


## Load Data

If the full raw files are available locally, load 2012 and 2013 data. If not, use the public sample dataset.


In [2]:
raw_2012_path = '../raw_data/2012_Data.csv'
raw_2013_path = '../raw_data/2013_Data.csv'
sample_path = '../data/sample/sample_luminatech_data.csv'

if os.path.exists(raw_2012_path) and os.path.exists(raw_2013_path):
    data_2012 = pd.read_csv(raw_2012_path, encoding='ISO-8859-1', engine='python', dtype={'customer_code': str})
    data_2013 = pd.read_csv(raw_2013_path, encoding='ISO-8859-1', engine='python', dtype={'customer_code': str})
    print('Loaded full local raw datasets')
else:
    data_2012 = pd.read_csv(sample_path, dtype={'customer_code': str})
    data_2013 = pd.DataFrame()
    print('Loaded public sample dataset')


Loaded full local raw datasets


In [3]:
data_2012.head()


,accounting_date,fiscal_year,fiscal_month,calendar_year,calendar_month,calendar_day,company_code,customer_code,customer_district_code,item_code,...,value_quantity,value_price_adjustment,currency,item_source_class,invoice_number,line_number,invoice_date,customer_order_number,order_date,dss_update_time
0,20120509,2012,11,2012,5,9,101,411800601,410,GENIE8WWWBC,...,84.0,0,AUD,NaN,2217887,1,20120509,2865354,20120509,49:58.7
1,20120216,2012,8,2012,2,16,101,361000403,300,GENIE8WWWBC,...,12.0,0,AUD,NaN,2185745,1,20120216,2833515,20120216,49:58.7
2,20120509,2012,11,2012,5,9,101,361000403,300,GENIE8WWWBC,...,12.0,0,AUD,NaN,2217807,1,20120509,2864857,20120508,49:58.7
3,20120518,2012,11,2012,5,18,101,565540415,500,GENIE8WWWBC,...,6.0,0,AUD,NaN,2222758,1,20120518,2869759,20120518,49:58.7
4,20120109,2012,7,2012,1,9,101,565540415,500,GENIE8WWWBC,...,6.0,0,AUD,NaN,2170374,1,20120109,2819189,20120109,49:58.7


In [4]:
print('2012 shape:', data_2012.shape)
print('2013 shape:', data_2013.shape)


2012 shape: (1037205, 41)
2013 shape: (951177, 41)


## Check Missing Values

This helps identify columns with missing values before cleaning.


In [5]:
data_2012.isnull().sum().sort_values(ascending=False).head(15)


item_source_class         1037205
fiscal_year                     0
fiscal_month                    0
calendar_year                   0
calendar_month                  0
calendar_day                    0
company_code                    0
customer_code                   0
accounting_date                 0
customer_district_code          0
item_code                       0
item_group_code                 0
business_area_code              0
item_type                       0
bonus_group_code                0
dtype: int64

## Drop Redundant Column

The original assignment removed `item_source_class`, so this notebook keeps that same cleaning step.


In [6]:
if 'item_source_class' in data_2012.columns:
    data_2012_cleaned = data_2012.drop(columns=['item_source_class'])
else:
    data_2012_cleaned = data_2012.copy()

if len(data_2013) > 0 and 'item_source_class' in data_2013.columns:
    data_2013_cleaned = data_2013.drop(columns=['item_source_class'])
else:
    data_2013_cleaned = data_2013.copy()


## Merge Datasets

Combine the yearly datasets into one transaction table.


In [7]:
if len(data_2013_cleaned) > 0:
    Dataset = pd.concat([data_2012_cleaned, data_2013_cleaned], ignore_index=True)
else:
    Dataset = data_2012_cleaned.copy()

Dataset.shape


(1988382, 40)

## Convert Date Columns

The original dates are stored as numbers like `20120131`, so we convert them to datetime.


In [8]:
date_columns = ['accounting_date', 'invoice_date', 'order_date']

for col in date_columns:
    if col in Dataset.columns:
        Dataset[col] = pd.to_datetime(Dataset[col].astype(str), format='%Y%m%d', errors='coerce')

Dataset[date_columns].head()


,accounting_date,invoice_date,order_date
0,2012-05-09,2012-05-09,2012-05-09
1,2012-02-16,2012-02-16,2012-02-16
2,2012-05-09,2012-05-09,2012-05-08
3,2012-05-18,2012-05-18,2012-05-18
4,2012-01-09,2012-01-09,2012-01-09


## Clean Currency and Business Rules

These are the simple cleaning rules used before analysis: standardise currency, remove missing currency rows, remove district code 100, and remove zero sales rows.


In [9]:
if 'currency' in Dataset.columns:
    Dataset['currency'] = Dataset['currency'].replace({'AUS': 'AUD', '   ': pd.NA})
    Dataset = Dataset.dropna(subset=['currency'])

if 'customer_district_code' in Dataset.columns:
    Dataset = Dataset[Dataset['customer_district_code'] != 100]

if 'value_sales' in Dataset.columns:
    Dataset = Dataset[Dataset['value_sales'] != 0].copy()

Dataset.shape


(1966082, 40)

## Create Profit and Processing-Time Fields

Profit supports profitability analysis. Time gap measures the number of days between order date and invoice date.


In [10]:
Dataset['profit'] = Dataset['value_sales'] - Dataset['value_cost']
Dataset['time_gap'] = (Dataset['invoice_date'] - Dataset['order_date']).dt.days

Dataset[['value_sales', 'value_cost', 'profit', 'time_gap']].head()


,value_sales,value_cost,profit,time_gap
0,218.40,178.1976,40.2024,0
1,38.28,25.4568,12.8232,0
2,40.20,25.4568,14.7432,1
3,20.10,12.7284,7.3716,0
4,19.14,12.7284,6.4116,0


## Save Cleaned Dataset

The cleaned file is saved locally for the next notebooks. This output is ignored by Git because it can be regenerated.


In [11]:
os.makedirs('../data/processed', exist_ok=True)
Dataset.to_csv('../data/processed/cleaned_luminatech_transactions.csv', index=False)

print('Saved cleaned dataset to ../data/processed/cleaned_luminatech_transactions.csv')


Saved cleaned dataset to ../data/processed/cleaned_luminatech_transactions.csv
